# Assignment: Similarity Matrices and Projection Heads

**Unit:** Core Multimodal Foundations | **Companion reading:** Lesson 2, Multimodal Representation and Latent Spaces
**Suggested time:** 1.5 to 2 hours

---

## What is this assignment trying to do?

Multimodal models like CLIP compare images and text by turning both into vectors and measuring the angle between them. In this assignment you will build that machinery yourself, from scratch, with small made-up feature vectors:

1. **Part 1:** Write cosine similarity.
2. **Part 2:** Build an image-by-text **similarity matrix** and use it to match images to captions.
3. **Part 3:** See why features from two *different* encoders cannot be compared directly.
4. **Part 4:** Train two **projection heads** with the **contrastive (InfoNCE) loss** and watch matching pairs line up.
5. **Part 5:** Experiment with the **temperature** and reflect on what you saw.

Everything runs on a CPU in under a minute. No GPU and no downloads are needed.

### How to work

- Fill in every cell marked `# TODO`. Each part ends with a **check** cell that tells you whether your code is right.
- Write your reflections as Python strings in the cells provided.
- Run every cell top to bottom before submitting.

---
## Part 0: Setup

Google Colab already has NumPy, PyTorch, and Matplotlib. Locally, install them with `pip install numpy torch matplotlib`.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 0
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
np.set_printoptions(precision=3, suppress=True)
print("Ready.")

---
## Part 1: Cosine Similarity

Cosine similarity measures how closely two vectors point in the same direction:

$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\lVert \mathbf{u} \rVert\, \lVert \mathbf{v} \rVert}$$

It is 1 for vectors pointing the same way, 0 for perpendicular vectors, and -1 for opposite vectors. Length does not matter.

### TODO 1: `cosine_similarity(u, v)`

Use `np.dot` and `np.linalg.norm`. Do not use any library function that computes cosine similarity for you.

In [ ]:
def cosine_similarity(u, v):
    """Cosine similarity between two 1-D NumPy arrays."""
    # TODO 1
    raise NotImplementedError("TODO 1: cosine_similarity")

In [ ]:
# Check Part 1 (PROVIDED)
assert abs(cosine_similarity(np.array([3., 4.]), np.array([4., 3.])) - 0.96) < 1e-9
assert abs(cosine_similarity(np.array([3., 4.]), np.array([-4., 3.]))) < 1e-9
assert abs(cosine_similarity(np.array([2., 0., 1.]), np.array([4., 0., 2.])) - 1.0) < 1e-9
assert abs(cosine_similarity(np.array([1., 2.]), np.array([-1., -2.])) + 1.0) < 1e-9
print("Part 1 passed.")

---
## Part 2: The Similarity Matrix

Imagine an encoder has already turned four images and four captions into vectors in a **shared** 4-dimensional space. To make the numbers readable, each dimension stands for one idea: `[animal, food, vehicle, outdoors]`. (Real embeddings have hundreds of dimensions and no human-readable meaning, but the math is identical.)

Row *i* of `IMAGES` and row *i* of `TEXTS` describe the same thing.

In [ ]:
# Dimensions: [animal, food, vehicle, outdoors]  (PROVIDED)
IMAGE_NAMES = ["dog on a beach", "slice of pizza", "red sports car", "mountain sunset"]
CAPTIONS    = ["a dog running on sand", "a slice of cheese pizza",
               "a fast red car", "sunset over the mountains"]

IMAGES = np.array([
    [0.9, 0.0, 0.0, 0.7],   # dog on a beach
    [0.0, 1.0, 0.0, 0.1],   # slice of pizza
    [0.0, 0.0, 0.9, 0.3],   # red sports car
    [0.1, 0.0, 0.0, 1.0],   # mountain sunset
])
TEXTS = np.array([
    [1.0, 0.0, 0.0, 0.4],   # a dog running on sand
    [0.0, 0.8, 0.0, 0.0],   # a slice of cheese pizza
    [0.0, 0.0, 1.0, 0.1],   # a fast red car
    [0.0, 0.0, 0.1, 0.9],   # sunset over the mountains
])

### TODO 2: `l2_normalize(X)` and `similarity_matrix(A, B)`

1. `l2_normalize(X)` divides **each row** of a 2-D array by that row's length, so every row has length 1. Hint: `np.linalg.norm(X, axis=1, keepdims=True)`.
2. `similarity_matrix(A, B)` returns the matrix `S` where `S[i, j]` is the cosine similarity between row *i* of `A` and row *j* of `B`. Normalize both, then use **one matrix multiplication** (`@`). No loops.

$$S = \hat{A}\, \hat{B}^{\top}$$

In [ ]:
def l2_normalize(X):
    """Scale every row of a 2-D array to length 1."""
    # TODO 2a
    raise NotImplementedError("TODO 2a: l2_normalize")


def similarity_matrix(A, B):
    """S[i, j] = cosine similarity between A[i] and B[j]. One matrix multiplication, no loops."""
    # TODO 2b
    raise NotImplementedError("TODO 2b: similarity_matrix")

In [ ]:
# Check Part 2 and plot (PROVIDED)
S = similarity_matrix(IMAGES, TEXTS)
assert S.shape == (4, 4)
for i in range(4):
    for j in range(4):
        assert abs(S[i, j] - cosine_similarity(IMAGES[i], TEXTS[j])) < 1e-9, (i, j)
print("Part 2 passed.\n")
print(S)

def plot_similarity(S, row_labels, col_labels, title):
    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    im = ax.imshow(S, cmap="viridis", vmin=-1 if S.min() < 0 else 0, vmax=1)
    ax.set_xticks(range(len(col_labels)), col_labels, rotation=40, ha="right")
    ax.set_yticks(range(len(row_labels)), row_labels)
    ax.set_xlabel("captions"); ax.set_ylabel("images"); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); plt.show()

plot_similarity(S, IMAGE_NAMES, CAPTIONS, "Cosine similarity, shared 4-D space")

### TODO 3: Retrieval in both directions

A similarity matrix answers two questions at once:

- **Image → text:** for each image (row), which caption scores highest?
- **Text → image:** for each caption (column), which image scores highest?

Write `retrieval_accuracy(S)`, which returns the fraction of rows whose highest score is on the diagonal (the true pair). Then use it on `S` and on `S.T`. Hint: `np.argmax(S, axis=1)`.

In [ ]:
def retrieval_accuracy(S):
    """Fraction of rows i whose best-scoring column is i."""
    # TODO 3
    raise NotImplementedError("TODO 3: retrieval_accuracy")

In [ ]:
# Check Part 2, retrieval (PROVIDED)
assert retrieval_accuracy(np.eye(3)) == 1.0
assert retrieval_accuracy(np.array([[0., 1.], [1., 0.]])) == 0.0
print("image -> text accuracy:", retrieval_accuracy(S))
print("text -> image accuracy:", retrieval_accuracy(S.T))
for i, name in enumerate(IMAGE_NAMES):
    print(f"  {name:16s} -> {CAPTIONS[int(np.argmax(S[i]))]!r}")

### Part 2 reflection

The best match for the dog photo scores below 1, and the dog image also has a positive score with "sunset over the mountains." Using the four dimensions, explain **why** those two numbers come out the way they do. Write your answer in the next cell.

In [ ]:
part2_reflection = """
TODO: your answer here
"""

---
## Part 3: Two Encoders, Two Spaces

Part 2 cheated: both sets of vectors were already in the same space. Real encoders do not cooperate like that. Below, we simulate 256 image–caption pairs. Each pair shares a hidden "meaning" vector `z`, but the image encoder and the text encoder express it completely differently:

- image features have **12** dimensions: `image = z @ A_img + noise`
- text features have **8** dimensions: `text = z @ A_txt + noise`

The first 128 pairs are for training and the last 128 for testing. This setup is PROVIDED; just run it.

In [ ]:
# Simulated encoders (PROVIDED)
N_TRAIN, N_TEST = 128, 128
LATENT, IMG_DIM, TXT_DIM = 8, 12, 8
NOISE = 0.6

Z = rng.normal(size=(N_TRAIN + N_TEST, LATENT))            # hidden shared meaning
A_img = rng.normal(size=(LATENT, IMG_DIM))                 # how the image encoder "sees" meaning
A_txt = rng.normal(size=(LATENT, TXT_DIM))                 # how the text encoder "writes" meaning
img_feats = Z @ A_img + NOISE * rng.normal(size=(N_TRAIN + N_TEST, IMG_DIM))
txt_feats = Z @ A_txt + NOISE * rng.normal(size=(N_TRAIN + N_TEST, TXT_DIM))

img_train, img_test = img_feats[:N_TRAIN], img_feats[N_TRAIN:]
txt_train, txt_test = txt_feats[:N_TRAIN], txt_feats[N_TRAIN:]
print("image features:", img_train.shape, " text features:", txt_train.shape)

Try to compare the raw features directly. It fails, because a 12-number vector and an 8-number vector have no dot product:

In [ ]:
try:
    similarity_matrix(img_test, txt_test)
except ValueError as e:
    print("ValueError:", e)

### TODO 4: Random projection heads

A **projection head** is a matrix that maps each encoder's output into a shared space of size `D`. Create two **random** projection matrices with NumPy, `W_img` with shape `(IMG_DIM, D)` and `W_txt` with shape `(TXT_DIM, D)`, project the **test** features, build the similarity matrix, and compute image-to-text retrieval accuracy.

With 128 test pairs, random guessing gets about 1/128, under 1%. Random projections make the shapes match, but do they make the spaces line up?

In [ ]:
D = 8

# TODO 4: random W_img (IMG_DIM x D) and W_txt (TXT_DIM x D) from rng.normal,
#         then S_random = similarity matrix of the projected test features
S_random = None
print("random projection accuracy:", retrieval_accuracy(S_random))

In [ ]:
# Check Part 3 (PROVIDED)
assert S_random is not None and S_random.shape == (N_TEST, N_TEST)
plot_similarity(S_random[:16, :16], [f"img {i}" for i in range(16)], [f"txt {i}" for i in range(16)],
                "Random projection heads (first 16 test pairs)")
print("Notice: no bright diagonal. Matching shapes is not the same as matching meaning.")

---
## Part 4: Train the Projection Heads with Contrastive Loss

Now we **learn** the projection heads. For a batch of *N* pairs, compute the *N × N* similarity matrix `S`, divide by a temperature `tau`, and treat every row as a classification problem whose correct answer is the diagonal:

$$\mathcal{L}_{I \to T} = -\frac{1}{N}\sum_{i} \log \frac{\exp(S_{ii}/\tau)}{\sum_{k}\exp(S_{ik}/\tau)}$$

Do the same for every column ($\mathcal{L}_{T \to I}$) and average the two. This symmetric **InfoNCE** loss is the one CLIP uses.

Cross-entropy already computes "softmax, then the negative log-probability of the correct class," so each direction is one call to `F.cross_entropy(logits, targets)`, where `targets = [0, 1, ..., N-1]`.

We switch to PyTorch here so it can compute gradients for us.

### TODO 5: `info_nce_loss(img_emb, txt_emb, tau)`

1. L2-normalize both embeddings: `F.normalize(x, dim=1)`.
2. `logits = img @ txt.T / tau`.
3. `targets = torch.arange(N)`.
4. Return the average of `F.cross_entropy(logits, targets)` and `F.cross_entropy(logits.T, targets)`.

In [ ]:
def info_nce_loss(img_emb, txt_emb, tau):
    """Symmetric InfoNCE loss for a batch of matching (image, text) embeddings."""
    # TODO 5
    raise NotImplementedError("TODO 5: info_nce_loss")

In [ ]:
# Check TODO 5 (PROVIDED)
_perfect = torch.eye(4)
assert info_nce_loss(_perfect, _perfect, tau=0.05).item() < 1e-3, "matching pairs should give a loss near 0"
_same = torch.ones(4, 3)
assert abs(info_nce_loss(_same, _same, tau=0.07).item() - np.log(4)) < 1e-4, "identical rows should give log(N)"
print("TODO 5 passed.")

### The training loop (PROVIDED)

`train_heads` creates two linear projection heads (12 → D and 8 → D), trains them on the training pairs with your loss, and returns them along with the loss history. Read it before running it: it is the same pattern as the training loops in the Foundations module.

In [ ]:
img_train_t = torch.tensor(img_train, dtype=torch.float32)
txt_train_t = torch.tensor(txt_train, dtype=torch.float32)
img_test_t = torch.tensor(img_test, dtype=torch.float32)
txt_test_t = torch.tensor(txt_test, dtype=torch.float32)


def train_heads(tau, D=8, steps=400, lr=0.05, seed=SEED):
    torch.manual_seed(seed)
    head_img = torch.nn.Linear(IMG_DIM, D, bias=False)
    head_txt = torch.nn.Linear(TXT_DIM, D, bias=False)
    opt = torch.optim.Adam(list(head_img.parameters()) + list(head_txt.parameters()), lr=lr)
    history = []
    for step in range(steps):
        loss = info_nce_loss(head_img(img_train_t), head_txt(txt_train_t), tau)
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(loss.item())
    return head_img, head_txt, history


def test_similarity(head_img, head_txt):
    with torch.no_grad():
        return similarity_matrix(head_img(img_test_t).numpy(), head_txt(txt_test_t).numpy())

### TODO 6: Train and evaluate

Train the heads with `tau = 0.07` (CLIP's starting temperature), compute the **test** similarity matrix with `test_similarity`, and report image-to-text and text-to-image retrieval accuracy. Store the matrix in `S_trained`.

In [ ]:
# TODO 6
S_trained = None
history = []

In [ ]:
# Check and plot Part 4 (PROVIDED)
assert S_trained is not None and S_trained.shape == (N_TEST, N_TEST)
assert retrieval_accuracy(S_trained) > 0.4, "trained heads should beat random projections by a wide margin"

plt.figure(figsize=(5, 3))
plt.plot(history); plt.xlabel("training step"); plt.ylabel("InfoNCE loss"); plt.title("Training loss")
plt.tight_layout(); plt.show()
plot_similarity(S_trained[:16, :16], [f"img {i}" for i in range(16)], [f"txt {i}" for i in range(16)],
                "Trained projection heads (first 16 test pairs)")
print(f"Random heads: {retrieval_accuracy(S_random):.1%}   Trained heads: {retrieval_accuracy(S_trained):.1%}")

### Part 4 reflection

The trained heads were only ever shown the **training** pairs, yet they match **test** pairs they never saw. What did the heads actually learn that lets them do this? Also, the trained heads still miss some test pairs. Using how the data was generated, explain why perfect accuracy is not possible here.

In [ ]:
part4_reflection = """
TODO: your answer here
"""

---
## Part 5: The Temperature Experiment

The temperature controls how sharply the softmax separates scores. In the lesson, a row of similarities `[0.30, 0.10, 0.05]` gave probability 0.385 for the right answer at `tau = 1`, but 0.921 at `tau = 0.07`.

### TODO 7: Sweep the temperature

For each `tau` in `TAUS`, train fresh heads, compute test image-to-text retrieval accuracy, and store it in the dictionary `results` (`results[tau] = accuracy`). The plotting code is provided.

In [ ]:
TAUS = [0.01, 0.07, 0.2, 0.5, 1.0, 5.0]
results = {}

# TODO 7


In [ ]:
# Check and plot Part 5 (PROVIDED)
assert set(results) == set(TAUS), "store one accuracy per temperature"
plt.figure(figsize=(5, 3))
plt.semilogx(list(results), list(results.values()), marker="o")
plt.xlabel("temperature tau (log scale)"); plt.ylabel("test retrieval accuracy")
plt.title("Effect of temperature"); plt.ylim(0, 1.05); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Part 5 reflection

Describe the pattern in your plot. Why does a very **high** temperature hurt? (Think about what happens to the softmax probabilities, and therefore to the gradients, when every logit is divided by a large number.) Did the smallest temperature behave the way you expected?

In [ ]:
part5_reflection = """
TODO: your answer here
"""

---
## Submitting

1. Make sure every `TODO` is complete and every check cell prints a passing message.
2. Run all cells from top to bottom (**Runtime → Run all** in Colab).
3. Download the notebook as a Python file (**File → Download → Download .py**) and rename it `firstname_lastname_similarity.py`.
4. Submit that `.py` file on Gradescope (see the assignment page on the course website for the link).